# P09 Skill を作る ― 読み込み係・発動の試験・最後まで

> **この教材に登場する会社・人物・製品・数値・文書はすべて架空のものです。実在の企業・団体・個人とは関係ありません。**

Messages API の上で「段階的に読む」を真似る読み込み係 (load_skill・read_skill_file・run_skill_script) を MCP のホストに足し、スキルが正しく発動するかと、月次レポートの数字が正解と合うかを確かめます。

- 上から順に実行します。0 のセルは準備です (Colab でも手元の Jupyter でも同じ)。
- API キーが無ければ、講師の実行記録を再生します (費用 0)。記録も無ければ mock で最後まで流れます (mock の数字は性能の証拠になりません)。
- 🔴 データはすべて架空の会社・文書です。

## 0. 準備

In [ ]:
# 0-1. 実行場所を見分けて、キットのフォルダに移る (Colab のときだけ取得とインストールをする)
import os, sys, subprocess
REPO_URL = "https://github.com/gucchi123/mcp-skills-colab.git"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    KIT = "/content/mcp-skills-kit"
    if not os.path.exists(KIT):
        subprocess.run(["git", "clone", "-q", REPO_URL, KIT], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{KIT}/requirements.txt"], check=True)
else:  # 手元: notebooks/ から開いた前提。キットの根 (common/ がある所) を探す
    KIT = os.path.abspath(".." if os.path.exists("../common/host.py") else ".")
os.chdir(KIT)
sys.path.insert(0, KIT)
print("実行場所:", "Colab" if IN_COLAB else "手元の Jupyter", "/ キットの根:", os.path.basename(KIT))

In [ ]:
# 0-2. Claude の呼び先を決める (キーは表示しない) と、ラボのモジュールを読めるようにする
from common.nb import choose_provider, safe
from common.kit import use_labs, utf8_console
PROVIDER = choose_provider(IN_COLAB)
use_labs(); utf8_console()

## 1. 読み込み係の道具と守り
モデルに見せる道具と、システムプロンプトのスキル一覧 (段階 1) を見ます。フォルダの外を読む・別のスキルのスクリプトを走らせる、といった呼び方は断ります (Claude を呼びません)。

In [ ]:
from run_build_skill import amain as build
with safe():
    r = await build(["--part", "loader"])

## 2. 発動の試験 (4 問)
問いごとに、最初に load_skill でどのスキルを読んだかを見ます。T-17 は「レポート」の語があっても月次ではない問いです。

In [ ]:
with safe():
    r = await build(["--part", "trigger", "--show"])

## 3. 説明文を曖昧にすると
説明文を「レポートを作る。」「文書を確認する。」に替えて同じ問いを流します。mock は説明文を読まないので差は出ません。本物のモデルで差を見ます。

In [ ]:
with safe():
    r = await build(["--part", "trigger", "--variant", "vague"])

## 4. 月次レポートを最後まで (2026-08)
スキルを読む → report サーバの道具を 3 つ → scripts/build_report.py → 書き方を読む → レポート。最後の文から数字を読み取り、正解と照合します。

In [ ]:
with safe():
    r = await build(["--month", "2026-08", "--show"])

## 5. 2 か月ぶんを照合する
数字 13 個が正解と合ったか、手順 (最初にスキルを読んだ・道具を 3 つ呼んだ・スクリプトが通った・書き方を読んだ・候補を書いた) が抜けていないかを並べます。

In [ ]:
with safe():
    r = await build(["--only", "2026-05,2026-09"])